In [1]:
from pathlib import Path
import sys
project_root = next(parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "src" / "ML_LC_Classifier").is_dir())
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

In [2]:
from ML_LC_Classifier import (
    load_and_split_training_data,
    select_features,
    tune_model,
    evaluate_model,
    classify_raster,
)

In [10]:
#define input path
landsat_input = r"C:\Agil_data\Project_EL\Landsat9_Final_Addtwi.tif"
sample_lc = r"C:\Agil_data\Project_EL\TrainingSamples_rev21.shp"
x_train, x_test, y_train, y_test = load_and_split_training_data(
    raster_path=landsat_input, 
    shapefile_path = sample_lc,
    class_field = 'LUCID',
    test_size = 0.4)

In [11]:
from xgboost import XGBClassifier
xgb_estimator = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    random_state=42,
    n_jobs=-1,
)
band_name = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B10', 
             'elevation', 'slope', 'TPILF', 'aspect', 'FlowA', 
             'TWI', 'brightness', 'greenness', 'wetness', 'TCA', 
             'NDVI', 'BUI', 'MNDWI', 'EVI', 'AWEI']
selection_result = select_features(x_train, y_train, x_test,
                                    estimator=xgb_estimator,
                                    feature_names = band_name,
                                    cv=5, 
                                    scoring="balanced_accuracy",
                                    min_features_to_select=10,)
x_train_selected = selection_result.X_train
x_test_selected = selection_result.X_test
selected_features = selection_result.selected_features

print(selected_features)
print(x_train_selected.shape)

['B1', 'B2', 'B3', 'B4', 'B7', 'B10', 'elevation', 'TPILF', 'brightness', 'greenness', 'wetness', 'BUI', 'MNDWI', 'EVI', 'AWEI']
(8756, 15)
